In [1]:
using Pkg                # Load Julia's built-in package manager module (Pkg)

Pkg.activate("..")       # Activate the Project.toml located one directory above
                         # (Allows notebooks/ to use the repository root environment)

using Horizons3DViewer2  # Load the custom package Horizons3DViewer2
                         # (This loads src/Horizons3DViewer2.jl and its modules)

using EphemerisSources   # Tools for accessing planetary ephemeris data
using AstroTime, Dates   # Time systems and date handling utilities
using GLMakie            # High-performance interactive visualization backend
using ProgressMeter      # Progress bar utilities for long computations
using LinearAlgebra      # Standard linear algebra operations (norm, dot, etc.)
using SPICE

  Activating project at `~/work/Horizons3DViewer2`


In [11]:
using GLMakie
using LinearAlgebra
using Printf
using Random

# ============================================================
# TDOA / FDOA による2次元位置・速度推定
#
# 状態:
#   X = [x_km, y_km, vx_ms, vy_ms]
#
# 観測:
#   z = [TDOA21_s, TDOA31_s, FDOA21_Hz, FDOA31_Hz]
#
# 真の状態は観測データ生成と答え合わせにのみ使用する。
# 推定器は、観測値、受信局位置、独立に設定した初期推定値だけを使う。
# ============================================================


# ============================================================
# 設定
# ============================================================

const C = 299_792_458.0          # 光速 [m/s]
const F_TX = 2.2e9               # 送信周波数 [Hz]

const VELOCITY_PLOT_SCALE = 0.5     # 位置平面での矢印倍率 [km/(m/s)]
const LOS_DISTANCE_TOLERANCE_KM = 1.0e-9

const POSITION_LIMIT_KM = 400000.0
const VELOCITY_LIMIT_MS = 10000.0

const CONFIDENCE_SCALE_68 = sqrt(2.279)
const CONFIDENCE_SCALE_95 = sqrt(5.991)

# スライダーを動かしても同じノイズ標本を使用する。
const NOISE_STANDARD_SAMPLE = randn(MersenneTwister(1234), 4)

# 状態変数の代表スケール。
# 数百 kmから40万 kmまでの広い位置範囲に対応するため、
# 初期推定位置の大きさから位置スケールを動的に設定する。
function make_state_scale(initial_state::AbstractVector)
    position_scale_km = max(norm(initial_state[1:2]), 1000.0)

    return Diagonal(Float64[
        position_scale_km,
        position_scale_km,
        1000.0,
        1000.0,
    ])
end

# whitened Jacobianの最小相対特異値がこの値以下なら、
# 実質的に非可観測または極端な悪条件と判定する。
const OBSERVABILITY_RELATIVE_SINGULAR_VALUE_THRESHOLD = 1.0e-10

satellite_altitude = 500.0 # km
#satellite_altitude = 1000.0 # km
#satellite_altitude = 36000.0 # km
#satellite_altitude = 360000.0 # km

# ============================================================
# 小さな変換関数
# ============================================================

point2d(v::AbstractVector) = Point2d(Float64(v[1]), Float64(v[2]))
vector2(p) = Float64[p[1], p[2]]

function state_from_position_velocity(position, velocity)
    return Float64[
        position[1],
        position[2],
        velocity[1],
        velocity[2],
    ]
end

function clamp_state(state::AbstractVector)
    return Float64[
        clamp(state[1], -POSITION_LIMIT_KM, POSITION_LIMIT_KM),
        clamp(state[2], -POSITION_LIMIT_KM, POSITION_LIMIT_KM),
        clamp(state[3], -VELOCITY_LIMIT_MS, VELOCITY_LIMIT_MS),
        clamp(state[4], -VELOCITY_LIMIT_MS, VELOCITY_LIMIT_MS),
    ]
end


# ============================================================
# 2次元仰角 El とその局所1σ
#
# 角度定義:
#   +x方向       :   0 deg
#   +y方向（真上）:  90 deg
#   -x方向       : 180 deg
#
# atan(y, x) を使用するため、y < 0 の場合は負の角度も返す。
# ============================================================

function elevation_angle_deg(position::AbstractVector)
    x = Float64(position[1])
    y = Float64(position[2])

    if !isfinite(x) || !isfinite(y) || hypot(x, y) < LOS_DISTANCE_TOLERANCE_KM
        return NaN
    end

    return rad2deg(atan(y, x))
end

function elevation_angle_deg(position)
    return elevation_angle_deg(Float64[position[1], position[2]])
end

"""
    elevation_sigma(position, position_covariance)

El = atan(y, x) のヤコビアン

    dEl/d[x,y] = [-y/r², x/r²]

を用いて、位置共分散を角度共分散へ写像する。
戻り値は局所1σで、rad / deg / arcsecを含む。
"""
function elevation_sigma(
    position::AbstractVector,
    position_covariance::AbstractMatrix,
)
    x = Float64(position[1])
    y = Float64(position[2])
    radius_squared = x^2 + y^2

    if !isfinite(radius_squared) ||
       radius_squared < LOS_DISTANCE_TOLERANCE_KM^2 ||
       !all(isfinite, position_covariance)
        return (
            valid = false,
            sigma_rad = NaN,
            sigma_deg = NaN,
            sigma_arcsec = NaN,
        )
    end

    angle_jacobian = Float64[-y / radius_squared, x / radius_squared]
    variance_rad2 =
        dot(angle_jacobian, position_covariance * angle_jacobian)

    if !isfinite(variance_rad2)
        return (
            valid = false,
            sigma_rad = NaN,
            sigma_deg = NaN,
            sigma_arcsec = NaN,
        )
    end

    sigma_rad = sqrt(max(variance_rad2, 0.0))
    sigma_deg = rad2deg(sigma_rad)

    return (
        valid = true,
        sigma_rad = sigma_rad,
        sigma_deg = sigma_deg,
        sigma_arcsec = sigma_deg * 3600.0,
    )
end


# ============================================================
# 物理モデル
# ============================================================

"""
    line_of_sight(object_position, station_position)

受信局から物体へ向かう単位視線ベクトルを返す。
物体と局がほぼ同位置の場合は `nothing` を返す。
"""
function line_of_sight(object_position, station_position)
    relative_position = vector2(object_position) - vector2(station_position)
    distance_km = norm(relative_position)

    if !isfinite(distance_km) || distance_km < LOS_DISTANCE_TOLERANCE_KM
        return nothing
    end

    return relative_position / distance_km
end

"""
    range_km(object_position, station_position)

物体と受信局の距離 [km]。
"""
function range_km(object_position, station_position)
    return hypot(
        object_position[1] - station_position[1],
        object_position[2] - station_position[2],
    )
end

"""
    calculate_measurements(position, velocity, stations)

3局について局1を基準とするTDOAとFDOAを計算する。

FDOAの符号定義:
    FDOA21 = f_rx,2 - f_rx,1
    FDOA31 = f_rx,3 - f_rx,1
"""
function calculate_measurements(
    object_position,
    object_velocity::AbstractVector,
    stations;
    f_tx::Real = F_TX,
    c::Real = C,
)
    ranges_m = zeros(3)
    range_rates_ms = zeros(3)
    received_frequencies_hz = zeros(3)

    for i in 1:3
        station = stations[i]
        los = line_of_sight(object_position, station)

        if los === nothing
            return (
                valid = false,
                ranges_m = fill(NaN, 3),
                range_rates_ms = fill(NaN, 3),
                received_frequencies_hz = fill(NaN, 3),
                tdoa21_s = NaN,
                tdoa31_s = NaN,
                fdoa21_hz = NaN,
                fdoa31_hz = NaN,
            )
        end

        ranges_m[i] = 1000.0 * range_km(object_position, station)
        range_rates_ms[i] = dot(los, object_velocity)
        received_frequencies_hz[i] = f_tx * (1.0 - range_rates_ms[i] / c)
    end

    # FDOAは約2.2 GHzの受信周波数同士を差し引かず、
    # レンジレート差から直接計算して桁落ちを避ける。
    fdoa21_hz = -f_tx / c * (range_rates_ms[2] - range_rates_ms[1])
    fdoa31_hz = -f_tx / c * (range_rates_ms[3] - range_rates_ms[1])

    return (
        valid = true,
        ranges_m = ranges_m,
        range_rates_ms = range_rates_ms,
        received_frequencies_hz = received_frequencies_hz,
        tdoa21_s = (ranges_m[2] - ranges_m[1]) / c,
        tdoa31_s = (ranges_m[3] - ranges_m[1]) / c,
        fdoa21_hz = fdoa21_hz,
        fdoa31_hz = fdoa31_hz,
    )
end

"""
    measurement_vector(state, stations)

状態 [x_km, y_km, vx_ms, vy_ms] から観測ベクトルを計算する。
モデルが無効な場合は `nothing` を返す。
"""
function measurement_vector(state::AbstractVector, stations)
    position = Point2d(state[1], state[2])
    velocity = view(state, 3:4)
    measurement = calculate_measurements(position, velocity, stations)

    measurement.valid || return nothing

    return Float64[
        measurement.tdoa21_s,
        measurement.tdoa31_s,
        measurement.fdoa21_hz,
        measurement.fdoa31_hz,
    ]
end


# ============================================================
# 観測誤差モデル
# ============================================================

"""
    differenced_measurement_covariance(sigma)

3局の独立誤差から、局1を共通基準とする2個の差分観測の
共分散行列を作る。
"""
function differenced_measurement_covariance(sigma::Real)
    return sigma^2 .* [
        2.0 1.0
        1.0 2.0
    ]
end

function observation_noise_model(
    sigma_toa_station::Real,
    sigma_freq_station::Real,
)
    tdoa_covariance =
        differenced_measurement_covariance(sigma_toa_station)
    fdoa_covariance =
        differenced_measurement_covariance(sigma_freq_station)

    observation_covariance = zeros(4, 4)
    observation_covariance[1:2, 1:2] .= tdoa_covariance
    observation_covariance[3:4, 3:4] .= fdoa_covariance

    observation_cholesky =
        cholesky(Symmetric(observation_covariance))

    observation_noise =
        observation_cholesky.L * NOISE_STANDARD_SAMPLE

    return (
        covariance = observation_covariance,
        cholesky = observation_cholesky,
        noise = observation_noise,
    )
end


# ============================================================
# 数値ヤコビアン
# ============================================================

"""
    numerical_measurement_jacobian(state, stations)

観測ベクトルの状態に対する4×4ヤコビアンを中央差分で計算する。
計算不能な場合は `nothing` を返す。
"""
function numerical_measurement_jacobian(
    state::AbstractVector,
    stations;
    relative_step::Real = 1.0e-7,
    minimum_position_step_km::Real = 0.01,
    minimum_velocity_step_ms::Real = 1.0e-3,
)
    # 遠方幾何では1 m程度の位置差分だと観測変化が丸め誤差に
    # 埋もれやすいため、状態量に比例した中央差分幅を使用する。
    steps = Float64[
        max(minimum_position_step_km, relative_step * max(abs(state[1]), 1.0)),
        max(minimum_position_step_km, relative_step * max(abs(state[2]), 1.0)),
        max(minimum_velocity_step_ms, relative_step * max(abs(state[3]), 1.0)),
        max(minimum_velocity_step_ms, relative_step * max(abs(state[4]), 1.0)),
    ]

    H = zeros(4, 4)

    for j in 1:4
        state_plus = copy(state)
        state_minus = copy(state)
        state_plus[j] += steps[j]
        state_minus[j] -= steps[j]

        z_plus = measurement_vector(state_plus, stations)
        z_minus = measurement_vector(state_minus, stations)

        if z_plus === nothing || z_minus === nothing
            return nothing
        end

        H[:, j] .= (z_plus - z_minus) / (2.0 * steps[j])
    end

    all(isfinite, H) || return nothing
    return H
end


# ============================================================
# 非線形重み付き最小二乗推定
# ============================================================

"""
    estimate_state_gauss_newton(observed_z, initial_state, stations)

減衰付きGauss–Newton法（Levenberg–Marquardt型）により、
TDOA/FDOA観測から [x, y, vx, vy] を同時推定する。

真の状態は使用しない。
"""
function estimate_state_gauss_newton(
    observed_z::AbstractVector,
    initial_state::AbstractVector,
    stations,
    observation_cholesky;
    max_iterations::Int = 40,
    step_tolerance::Real = 1.0e-7,
    cost_tolerance::Real = 1.0e-10,
    initial_damping::Real = 1.0e-2,
)
    invalid_result(message; state = clamp_state(initial_state), iterations = 0) = (
        valid = false,
        converged = false,
        message = message,
        state = Float64.(state),
        covariance = fill(NaN, 4, 4),
        position_covariance = fill(NaN, 2, 2),
        velocity_covariance = fill(NaN, 2, 2),
        residual = fill(NaN, 4),
        normalized_residual = fill(NaN, 4),
        cost = Inf,
        iterations = iterations,
        jacobian = fill(NaN, 4, 4),
        scaled_information_condition = Inf,
        singular_values = fill(NaN, 4),
        relative_singular_values = fill(NaN, 4),
        geometry_reliable = false,
    )

    length(observed_z) == 4 || return invalid_result("Observation size must be 4")
    length(initial_state) == 4 || return invalid_result("State size must be 4")
    all(isfinite, observed_z) || return invalid_result("Observation contains NaN/Inf")

    state = clamp_state(Float64.(initial_state))
    state_scale = make_state_scale(state)
    damping = Float64(initial_damping)
    converged = false
    message = "Maximum iterations reached"
    previous_cost = Inf
    iteration_count = 0

    for iteration in 1:max_iterations
        iteration_count = iteration

        predicted_z = measurement_vector(state, stations)
        predicted_z === nothing && return invalid_result(
            "Invalid geometry at current estimate";
            state = state,
            iterations = iteration,
        )

        residual = observed_z - predicted_z
        normalized_residual = observation_cholesky.L \ residual
        cost = 0.5 * dot(normalized_residual, normalized_residual)

        H = numerical_measurement_jacobian(state, stations)
        H === nothing && return invalid_result(
            "Jacobian could not be evaluated";
            state = state,
            iterations = iteration,
        )

        # 状態を代表スケールで無次元化したヤコビアン
        H_scaled = H * state_scale
        H_whitened = observation_cholesky.L \ H_scaled

        normal_matrix = H_whitened' * H_whitened
        gradient = H_whitened' * normalized_residual

        accepted = false
        scaled_step = zeros(4)
        candidate_cost = cost

        # 悪条件時にも動けるよう、減衰を増減しながら試行する。
        for _ in 1:12
            damped_matrix = normal_matrix + damping * I

            try
                scaled_step = damped_matrix \ gradient
            catch
                damping *= 10.0
                continue
            end

            all(isfinite, scaled_step) || begin
                damping *= 10.0
                continue
            end

            candidate_state = clamp_state(state + state_scale * scaled_step)
            candidate_z = measurement_vector(candidate_state, stations)

            if candidate_z === nothing
                damping *= 10.0
                continue
            end

            candidate_residual = observed_z - candidate_z
            candidate_normalized_residual =
                observation_cholesky.L \ candidate_residual
            candidate_cost =
                0.5 * dot(candidate_normalized_residual, candidate_normalized_residual)

            if isfinite(candidate_cost) && candidate_cost < cost
                state = candidate_state
                damping = max(damping / 3.0, 1.0e-12)
                accepted = true
                break
            end

            damping *= 10.0
        end

        if !accepted
            message = "No cost-reducing step found"
            break
        end

        physical_step = state_scale * scaled_step
        relative_step = norm(physical_step) / max(norm(state), 1.0)
        relative_cost_change = abs(cost - candidate_cost) / max(cost, 1.0)

        if relative_step < step_tolerance
            converged = true
            message = "Converged: state step tolerance"
            break
        elseif relative_cost_change < cost_tolerance
            converged = true
            message = "Converged: cost tolerance"
            break
        elseif candidate_cost > previous_cost && iteration > 2
            message = "Cost stopped improving"
            break
        end

        previous_cost = candidate_cost
    end

    predicted_z = measurement_vector(state, stations)
    predicted_z === nothing && return invalid_result(
        "Final state is invalid";
        state = state,
        iterations = iteration_count,
    )

    residual = observed_z - predicted_z
    normalized_residual = observation_cholesky.L \ residual
    final_cost = 0.5 * dot(normalized_residual, normalized_residual)

    H = numerical_measurement_jacobian(state, stations)
    H === nothing && return invalid_result(
        "Final Jacobian is invalid";
        state = state,
        iterations = iteration_count,
    )

    H_scaled = H * state_scale
    H_whitened = observation_cholesky.L \ H_scaled

    # 情報行列 H' H を直接擬似逆にすると条件数が二乗され、
    # 弱観測方向が数値的に失われやすい。
    # そのため、whitened Jacobianを直接SVD分解して共分散を求める。
    svd_H = svd(H_whitened)
    singular_values = svd_H.S

    maximum_singular_value = maximum(singular_values)

    relative_singular_values = if maximum_singular_value > 0.0
        singular_values ./ maximum_singular_value
    else
        zeros(length(singular_values))
    end

    geometry_reliable =
        minimum(relative_singular_values) >
        OBSERVABILITY_RELATIVE_SINGULAR_VALUE_THRESHOLD

    # 完全にゼロと判定された弱観測方向をゼロ分散にしないための下限。
    # これは極端に大きな分散を有限値として表現するための数値的上限である。
    relative_singular_value_floor = 1.0e-12
    singular_value_floor =
        max(maximum_singular_value * relative_singular_value_floor, eps(Float64))

    inverse_squared_singular_values = [
        1.0 / max(s, singular_value_floor)^2
        for s in singular_values
    ]

    scaled_covariance =
        svd_H.V *
        Diagonal(inverse_squared_singular_values) *
        svd_H.V'

    state_covariance =
        Matrix(state_scale * scaled_covariance * state_scale')
    state_covariance =
        (state_covariance + state_covariance') / 2

    geometry_condition = if minimum(singular_values) <= eps(Float64)
        Inf
    else
        maximum_singular_value / minimum(singular_values)
    end

    valid = all(isfinite, state) && all(isfinite, state_covariance)

    return (
        valid = valid,
        converged = converged,
        message = message,
        state = state,
        covariance = state_covariance,
        position_covariance = state_covariance[1:2, 1:2],
        velocity_covariance = state_covariance[3:4, 3:4],
        residual = residual,
        normalized_residual = normalized_residual,
        cost = final_cost,
        iterations = iteration_count,
        jacobian = H,
        scaled_information_condition = geometry_condition,
        singular_values = singular_values,
        relative_singular_values = relative_singular_values,
        geometry_reliable = geometry_reliable,
    )
end


# ============================================================
# 位置既知としてFDOAから速度を求める補助推定
# ============================================================

"""
    estimate_velocity_from_fdoa(position, stations, fdoa21, fdoa31)

与えられた位置における2本のFDOA制約から速度を求める。
このコードでは、真位置ではなく非線形推定で得た位置を使用する。
"""
function estimate_velocity_from_fdoa(
    object_position,
    stations,
    fdoa21_hz::Real,
    fdoa31_hz::Real;
    determinant_tolerance::Real = 1.0e-8,
)
    u1 = line_of_sight(object_position, stations[1])
    u2 = line_of_sight(object_position, stations[2])
    u3 = line_of_sight(object_position, stations[3])

    if u1 === nothing || u2 === nothing || u3 === nothing
        return (
            valid = false,
            velocity_ms = [NaN, NaN],
            determinant = NaN,
            condition_number = Inf,
        )
    end

    A = [
        (u2 - u1)[1] (u2 - u1)[2]
        (u3 - u1)[1] (u3 - u1)[2]
    ]

    b = -C / F_TX .* Float64[fdoa21_hz, fdoa31_hz]
    determinant = det(A)

    if !isfinite(determinant) || abs(determinant) < determinant_tolerance
        return (
            valid = false,
            velocity_ms = [NaN, NaN],
            determinant = determinant,
            condition_number = Inf,
        )
    end

    velocity = A \ b

    return (
        valid = all(isfinite, velocity),
        velocity_ms = velocity,
        determinant = determinant,
        condition_number = cond(A),
    )
end


# ============================================================
# 描画用補助関数
# ============================================================

function confidence_ellipse_points(
    center::AbstractVector,
    covariance::AbstractMatrix,
    scale::Real;
    number_of_points::Int = 200,
)
    if !all(isfinite, center) || !all(isfinite, covariance)
        return [Point2d(NaN, NaN)]
    end

    covariance_symmetric = Symmetric((covariance + covariance') / 2)
    eig = eigen(covariance_symmetric)
    eigenvalues = max.(eig.values, 0.0)
    transformation = eig.vectors * Diagonal(sqrt.(eigenvalues))

    return [
        begin
            unit_point = [cos(angle), sin(angle)]
            ellipse_point = center + scale * transformation * unit_point
            Point2d(ellipse_point[1], ellipse_point[2])
        end
        for angle in range(0.0, 2π; length = number_of_points)
    ]
end

"""
    fdoa_constraint(position, reference_station, target_station, fdoa)

速度平面上の直線 a'v = b を返す。
"""
function fdoa_constraint(
    position,
    reference_station,
    target_station,
    fdoa_hz::Real,
)
    u_reference = line_of_sight(position, reference_station)
    u_target = line_of_sight(position, target_station)

    if u_reference === nothing || u_target === nothing
        return (valid = false, a = [NaN, NaN], b = NaN)
    end

    return (
        valid = true,
        a = u_target - u_reference,
        b = -C / F_TX * fdoa_hz,
    )
end

function constraint_plot_data(constraint, vx_grid)
    if !constraint.valid
        return (
            vy = fill(NaN, length(vx_grid)),
            vertical_x = NaN,
        )
    end

    ax, ay = constraint.a
    b = constraint.b

    if abs(ay) > 1.0e-10
        return (
            vy = [(b - ax * vx) / ay for vx in vx_grid],
            vertical_x = NaN,
        )
    elseif abs(ax) > 1.0e-10
        return (
            vy = fill(NaN, length(vx_grid)),
            vertical_x = b / ax,
        )
    end

    return (
        vy = fill(NaN, length(vx_grid)),
        vertical_x = NaN,
    )
end



# ============================================================
# 解析的なTDOA双曲線
# ============================================================

"""
    analytic_tdoa_hyperbola(reference_station, target_station, delta_range_km;
                            plot_extent_km, number_of_points=2001)

距離差

    |p - target_station| - |p - reference_station| = delta_range_km

を満たすTDOA双曲線を、標準形から解析的にパラメータ表示する。

戻り値は `lines!` に直接渡せる `Vector{Point2d}`。

注意:
- `abs(delta_range_km) < station_distance` のとき通常の双曲線。
- `delta_range_km ≈ 0` のとき垂直二等分線へ退化する。
- `abs(delta_range_km) > station_distance` のとき実解は存在しない。
"""
function analytic_tdoa_hyperbola(
    reference_station,
    target_station,
    delta_range_km::Real;
    plot_extent_km::Real,
    number_of_points::Int = 2001,
)
    s1 = vector2(reference_station)
    s2 = vector2(target_station)
    baseline = s2 - s1
    station_distance = norm(baseline)

    if !isfinite(station_distance) ||
       station_distance < LOS_DISTANCE_TOLERANCE_KM ||
       !isfinite(delta_range_km)
        return [Point2d(NaN, NaN)]
    end

    # 三角不等式より、距離差の絶対値は局間距離を超えられない。
    feasibility_tolerance = 100.0 * eps(Float64) * max(station_distance, 1.0)
    if abs(delta_range_km) > station_distance + feasibility_tolerance
        return [Point2d(NaN, NaN)]
    end

    center = (s1 + s2) / 2.0
    baseline_unit = baseline / station_distance
    transverse_unit = Float64[-baseline_unit[2], baseline_unit[1]]

    # 距離差が0の場合、双曲線は両局の垂直二等分線へ退化する。
    zero_difference_tolerance =
        100.0 * eps(Float64) * max(station_distance, 1.0)

    if abs(delta_range_km) <= zero_difference_tolerance
        extent = max(Float64(plot_extent_km), station_distance)
        return [
            Point2d((center - extent * transverse_unit)...),
            Point2d((center + extent * transverse_unit)...),
        ]
    end

    # 標準双曲線 x²/a² - y²/b² = 1
    # 焦点半距離 c、距離差の半分 a、b² = c² - a²。
    c_half = station_distance / 2.0
    a = abs(Float64(delta_range_km)) / 2.0
    b_squared = max(c_half^2 - a^2, 0.0)
    b = sqrt(b_squared)

    # |p-s2| - |p-s1| = Δr の符号に対応する枝を選ぶ。
    # 基線座標の正側（s2側）では r2-r1 < 0 となる。
    branch_sign = -sign(Float64(delta_range_km))

    # 表示範囲を十分に覆うτ範囲を自動設定する。
    target_extent = max(4.0 * Float64(plot_extent_km), 2.0 * station_distance)
    tau_from_x = acosh(max(target_extent / max(a, eps(Float64)), 1.0))
    tau_from_y = b > eps(Float64) ? asinh(target_extent / b) : 0.0
    tau_max = clamp(max(tau_from_x, tau_from_y), 1.0, 15.0)

    tau_values = range(-tau_max, tau_max; length = number_of_points)

    return [
        begin
            xi = branch_sign * a * cosh(tau)
            eta = b * sinh(tau)
            point = center + xi * baseline_unit + eta * transverse_unit
            Point2d(point[1], point[2])
        end
        for tau in tau_values
    ]
end

# ============================================================
# FigureとAxes
# ============================================================

fig = Figure(size = (2100, 1250))


area_length = satellite_altitude/2*1.2

ax_position = Axis(
    fig[1:2, 1:2],
    xlabel = "x [km]",
    ylabel = "y [km]",
    title = "Position plane",
    aspect = DataAspect(),
)

xlims!(ax_position, -area_length, area_length)
ylims!(ax_position, -area_length/10, area_length*2)

ax_velocity = Axis(
    fig[1:2, 3:4],
    xlabel = "vx [m/s]",
    ylabel = "vy [m/s]",
    title = "Velocity plane",
    aspect = DataAspect(),
)

xlims!(ax_velocity, -1200, 1200)
ylims!(ax_velocity, -1200, 1200)


# ============================================================
# 受信局スライダー
# ============================================================

station_slider_grid = SliderGrid(
    fig[4, 1:2],
    (
        label = "Station 1: x [km]",
        range = -1000:10.0:1000,
        startvalue = 0.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station 1: y [km]",
        range = 0.0:10.0:area_length*2,
        startvalue = 0.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station 2: x [km]",
        range = -1000:10.0:1000,
        startvalue = -500.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station 2: y [km]",
        range = 0.0:10.0:area_length*2,
        startvalue = 0.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station 3: x [km]",
        range = -1000:10.0:1000,
        startvalue = 500.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station 3: y [km]",
        range = 0.0:10.0:area_length*2,
        startvalue = 0.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station TDOA σ [ns]",
        range = 1.0:1.0:100.0,
        startvalue = 1.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Station FDOA σ [Hz]",
        range = 0.01:0.01:1.0,
        startvalue = 0.01,
        format = x -> @sprintf("%.2f", x),
    ),
    width = 430,
)

station_slider_values = [s.value for s in station_slider_grid.sliders]

sigma_toa_station = lift(station_slider_values[7]) do sigma_nanoseconds
    # スライダー値 [ns] を内部計算用の秒 [s] に変換する。
    sigma_nanoseconds * 1.0e-9
end

sigma_freq_station = station_slider_values[8]

observation_model = lift(
    sigma_toa_station,
    sigma_freq_station,
) do sigma_toa, sigma_freq
    observation_noise_model(sigma_toa, sigma_freq)
end

station1 = lift(station_slider_values[1], station_slider_values[2]) do x, y
    Point2d(x, y)
end

station2 = lift(station_slider_values[3], station_slider_values[4]) do x, y
    Point2d(x, y)
end

station3 = lift(station_slider_values[5], station_slider_values[6]) do x, y
    Point2d(x, y)
end

stations = lift(station1, station2, station3) do s1, s2, s3
    (s1, s2, s3)
end


# ============================================================
# 真の物体状態スライダー
# 真値は観測生成と答え合わせだけに使用する。
# 対象物体の既定速度は 1 km/s = 1000 m/s とする。
# ============================================================

true_state_slider_grid = SliderGrid(
    fig[4, 3:4],
    (
        label = "True object: x [km]",
        range = -area_length:5.0:area_length,
        startvalue = area_length*0.1,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "True object: y [km]",
        range = 0.0:5.0:area_length*2,
        startvalue = satellite_altitude,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "True object: speed [m/s]",
        range = 0.0:10.0:2000.0,
        startvalue = 1000.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "True object: angle [deg]",
        range = -180.0:1.0:180.0,
        startvalue = -18.0,
        format = x -> @sprintf("%.0f", x),
    ),
    #width = 430,
        (
        label = "Initial guess: x [km]",
        range = -area_length:5.0:area_length,
        startvalue = area_length*0.1,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: y [km]",
        range = 0.0:5.0:area_length*2,
        startvalue = area_length*2*0.8,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: vx [m/s]",
        range = -2000.0:10.0:2000.0,
        startvalue = 950.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: vy [m/s]",
        range = -2000.0:10.0:2000.0,
        startvalue = -310.0,
        format = x -> @sprintf("%.0f", x),
    ),
    width = 430,
)

true_slider_values = [s.value for s in true_state_slider_grid.sliders]

true_position = lift(true_slider_values[1], true_slider_values[2]) do x, y
    Point2d(x, y)
end

true_velocity = lift(true_slider_values[3], true_slider_values[4]) do speed, angle_deg
    Float64[speed * cosd(angle_deg), speed * sind(angle_deg)]
end

true_state = lift(true_position, true_velocity) do position, velocity
    state_from_position_velocity(position, velocity)
end


# ============================================================
# 推定初期値スライダー
# 真値とは独立に設定する。
# ============================================================
#=

initial_guess_slider_grid = SliderGrid(
    fig[5, 1:4],
    (
        label = "Initial guess: x [km]",
        range = -400.0:5.0:400.0,
        startvalue = 40.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: y [km]",
        range = -350.0:5.0:400.0,
        startvalue = 80.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: vx [m/s]",
        range = -300.0:5.0:300.0,
        startvalue = 60.0,
        format = x -> @sprintf("%.0f", x),
    ),
    (
        label = "Initial guess: vy [m/s]",
        range = -300.0:5.0:300.0,
        startvalue = 20.0,
        format = x -> @sprintf("%.0f", x),
    ),
    width = 950,
)
=#

initial_state = lift(
    true_slider_values[5],
    true_slider_values[6],
    true_slider_values[7],
    true_slider_values[8],
) do x, y, vx, vy
    Float64[x, y, vx, vy]
end


# ============================================================
# 観測生成
# ============================================================

true_measurements = lift(true_position, true_velocity, stations) do position, velocity, st
    calculate_measurements(position, velocity, st)
end

observed_measurement_vector = lift(
    true_measurements,
    observation_model,
) do measurement, noise_model
    if !measurement.valid
        return fill(NaN, 4)
    end

    true_z = Float64[
        measurement.tdoa21_s,
        measurement.tdoa31_s,
        measurement.fdoa21_hz,
        measurement.fdoa31_hz,
    ]

    return true_z + noise_model.noise
end

noisy_measurements = lift(true_measurements, observed_measurement_vector) do measurement, z
    return (
        valid = measurement.valid && all(isfinite, z),
        ranges_m = measurement.ranges_m,
        range_rates_ms = measurement.range_rates_ms,
        received_frequencies_hz = measurement.received_frequencies_hz,
        tdoa21_s = z[1],
        tdoa31_s = z[2],
        fdoa21_hz = z[3],
        fdoa31_hz = z[4],
    )
end


# ============================================================
# 非線形状態推定
# ============================================================

state_solution = lift(
    observed_measurement_vector,
    initial_state,
    stations,
    observation_model,
) do observed_z, initial, st, noise_model
    estimate_state_gauss_newton(
        observed_z,
        initial,
        st,
        noise_model.cholesky,
    )
end

estimated_position = lift(state_solution) do solution
    solution.valid ? Point2d(solution.state[1], solution.state[2]) : Point2d(NaN, NaN)
end

estimated_velocity = lift(state_solution) do solution
    solution.valid ? solution.state[3:4] : Float64[NaN, NaN]
end

solution_visible = lift(state_solution) do solution
    solution.valid
end

ellipse_visible = lift(state_solution) do solution
    solution.valid && solution.geometry_reliable
end


# ============================================================
# 推定位置を既知としたFDOA速度推定
# ============================================================

fdoa_velocity_solution = lift(
    estimated_position,
    stations,
    noisy_measurements,
) do position, st, measurement
    if !measurement.valid || !all(isfinite, Tuple(position))
        return (
            valid = false,
            velocity_ms = [NaN, NaN],
            determinant = NaN,
            condition_number = Inf,
        )
    end

    estimate_velocity_from_fdoa(
        position,
        st,
        measurement.fdoa21_hz,
        measurement.fdoa31_hz,
    )
end

fdoa_velocity_visible = lift(fdoa_velocity_solution) do solution
    solution.valid
end


# ============================================================
# TDOA双曲線（解析的パラメータ表示）
# ============================================================

range_difference21_km = lift(noisy_measurements) do measurement
    C * measurement.tdoa21_s / 1000.0
end

range_difference31_km = lift(noisy_measurements) do measurement
    C * measurement.tdoa31_s / 1000.0
end

# contour!による格子補間は使わず、
# 双曲線の標準形から直接Point2d列を生成する。
hyperbola_plot_extent_km = 4.0 * area_length

hyperbola21_points = lift(
    station1,
    station2,
    range_difference21_km,
) do s1, s2, delta_range
    analytic_tdoa_hyperbola(
        s1,
        s2,
        delta_range;
        plot_extent_km = hyperbola_plot_extent_km,
        number_of_points = 3001,
    )
end

hyperbola31_points = lift(
    station1,
    station3,
    range_difference31_km,
) do s1, s3, delta_range
    analytic_tdoa_hyperbola(
        s1,
        s3,
        delta_range;
        plot_extent_km = hyperbola_plot_extent_km,
        number_of_points = 3001,
    )
end

lines!(
    ax_position,
    hyperbola21_points;
    color = :cyan,
    linewidth = 3,
    label = "TDOA: S2 - S1",
)

lines!(
    ax_position,
    hyperbola31_points;
    color = :magenta,
    linewidth = 3,
    label = "TDOA: S3 - S1",
)

# ============================================================
# 位置平面の描画
# ============================================================

station_points = lift(stations) do st
    collect(st)
end

scatter!(
    ax_position,
    station_points;
    color = [:red, :green, :blue],
    markersize = 22,
)

for i in 1:3
    station_i = lift(stations) do st
        st[i]
    end

    text!(
        ax_position,
        station_i;
        text = "S$i",
        offset = (12, 12),
        fontsize = 20,
        font = :bold,
    )
end

true_position_marker = lift(true_position) do position
    [position]
end

initial_position_marker = lift(initial_state) do state
    [Point2d(state[1], state[2])]
end

estimated_position_marker = lift(estimated_position) do position
    [position]
end

scatter!(
    ax_position,
    true_position_marker;
    color = :lime,
    strokecolor = :black,
    strokewidth = 2,
    markersize = 25,
    label = "True position",
)

scatter!(
    ax_position,
    estimated_position_marker;
    color = :orange,
    strokecolor = :black,
    strokewidth = 2,
    marker = :star5,
    markersize = 28,
    visible = solution_visible,
    label = "least-squares estimated position",
)

true_velocity_arrow = lift(true_velocity) do velocity
    [Vec2d(
        VELOCITY_PLOT_SCALE * velocity[1],
        VELOCITY_PLOT_SCALE * velocity[2],
    )]
end

estimated_velocity_arrow = lift(estimated_velocity) do velocity
    if all(isfinite, velocity)
        [Vec2d(
            VELOCITY_PLOT_SCALE * velocity[1],
            VELOCITY_PLOT_SCALE * velocity[2],
        )]
    else
        [Vec2d(0.0, 0.0)]
    end
end

#=
arrows2d!(
    ax_position,
    true_position_marker,
    true_velocity_arrow;
    color = :lime,
)

arrows2d!(
    ax_position,
    estimated_position_marker,
    estimated_velocity_arrow;
    color = :orange,
    visible = solution_visible,
)
=#

position_ellipse_68 = lift(state_solution) do solution
    confidence_ellipse_points(
        solution.state[1:2],
        solution.position_covariance,
        CONFIDENCE_SCALE_68,
    )
end

position_ellipse_95 = lift(state_solution) do solution
    confidence_ellipse_points(
        solution.state[1:2],
        solution.position_covariance,
        CONFIDENCE_SCALE_95,
    )
end

lines!(
    ax_position,
    position_ellipse_95;
    color = :orange,
    linewidth = 2,
    linestyle = :dash,
    visible = ellipse_visible,
    label = "Position 95% local confidence",
)

lines!(
    ax_position,
    position_ellipse_68;
    color = :orange,
    linewidth = 2,
    visible = ellipse_visible,
    label = "Position 68% local confidence",
)

#=
scatter!(
    ax_position,
    initial_position_marker;
    color = :gray,
    strokecolor = :black,
    strokewidth = 1,
    marker = :diamond,
    markersize = 20,
    label = "Initial position guess",
)
=#

# ============================================================
# FDOA速度制約直線
# 推定位置における視線ベクトルを使用する。
# ============================================================

vx_grid = range(-2000.0, 2000.0, length = 2001)

fdoa_constraint_lines = lift(
    estimated_position,
    stations,
    noisy_measurements,
) do position, st, measurement
    constraint21 = fdoa_constraint(
        position,
        st[1],
        st[2],
        measurement.fdoa21_hz,
    )

    constraint31 = fdoa_constraint(
        position,
        st[1],
        st[3],
        measurement.fdoa31_hz,
    )

    return (
        line21 = constraint_plot_data(constraint21, vx_grid),
        line31 = constraint_plot_data(constraint31, vx_grid),
    )
end

vy21 = lift(fdoa_constraint_lines) do data
    data.line21.vy
end

vy31 = lift(fdoa_constraint_lines) do data
    data.line31.vy
end

vertical21_position = lift(fdoa_constraint_lines) do data
    isfinite(data.line21.vertical_x) ? [data.line21.vertical_x] : [0.0]
end

vertical31_position = lift(fdoa_constraint_lines) do data
    isfinite(data.line31.vertical_x) ? [data.line31.vertical_x] : [0.0]
end

vertical21_visible = lift(fdoa_constraint_lines, solution_visible) do data, visible
    visible && isfinite(data.line21.vertical_x)
end

vertical31_visible = lift(fdoa_constraint_lines, solution_visible) do data, visible
    visible && isfinite(data.line31.vertical_x)
end

lines!(
    ax_velocity,
    vx_grid,
    vy21;
    color = :cyan,
    linewidth = 3,
    visible = solution_visible,
    label = "FDOA: S2 - S1",
)

lines!(
    ax_velocity,
    vx_grid,
    vy31;
    color = :magenta,
    linewidth = 3,
    visible = solution_visible,
    label = "FDOA: S3 - S1",
)

vlines!(
    ax_velocity,
    vertical21_position;
    color = :cyan,
    linewidth = 3,
    visible = vertical21_visible,
)

vlines!(
    ax_velocity,
    vertical31_position;
    color = :magenta,
    linewidth = 3,
    visible = vertical31_visible,
)

# ============================================================
# 表示を常に推定を中心にする
# ============================================================
#=
velocity_range = 10.0

lift(estimated_velocity) do velocity
    limits!(
        ax_velocity,
        velocity[1] - velocity_range,
        velocity[1] + velocity_range,
        velocity[2] - velocity_range,
        velocity[2] + velocity_range,
    )
end

position_range = 10.0

lift(estimated_position) do position
    limits!(
        ax_position,
        position[1] - position_range,
        position[1] + position_range,
        position[2] - position_range,
        position[2] + position_range,
    )
end
=#

# ============================================================
# 速度平面の状態点と信頼楕円
# ============================================================

true_velocity_point = lift(true_velocity) do velocity
    [Point2d(velocity[1], velocity[2])]
end

initial_velocity_point = lift(initial_state) do state
    [Point2d(state[3], state[4])]
end

estimated_velocity_point = lift(estimated_velocity) do velocity
    [Point2d(velocity[1], velocity[2])]
end

fdoa_velocity_point = lift(fdoa_velocity_solution) do solution
    solution.valid ?
        [Point2d(solution.velocity_ms[1], solution.velocity_ms[2])] :
        [Point2d(NaN, NaN)]
end

scatter!(
    ax_velocity,
    true_velocity_point;
    color = :lime,
    strokecolor = :black,
    strokewidth = 2,
    markersize = 24,
    label = "True velocity",
)

scatter!(
    ax_velocity,
    estimated_velocity_point;
    color = :orange,
    strokecolor = :black,
    strokewidth = 2,
    marker = :star5,
    markersize = 30,
    visible = solution_visible,
    label = "least-squares estimated velocity",
)

#=
scatter!(
    ax_velocity,
    fdoa_velocity_point;
    color = :magenta,
    strokecolor = :black,
    strokewidth = 2,
    marker = :circle,
    markersize = 19,
    visible = fdoa_velocity_visible,
    label = "FDOA-only at estimated position",
)
=#

velocity_ellipse_68 = lift(state_solution) do solution
    confidence_ellipse_points(
        solution.state[3:4],
        solution.velocity_covariance,
        CONFIDENCE_SCALE_68,
    )
end

velocity_ellipse_95 = lift(state_solution) do solution
    confidence_ellipse_points(
        solution.state[3:4],
        solution.velocity_covariance,
        CONFIDENCE_SCALE_95,
    )
end

lines!(
    ax_velocity,
    velocity_ellipse_95;
    color = :orange,
    linewidth = 2,
    linestyle = :dash,
    visible = ellipse_visible,
    label = "Velocity 95% local confidence",
)

lines!(
    ax_velocity,
    velocity_ellipse_68;
    color = :orange,
    linewidth = 2,
    visible = ellipse_visible,
    label = "Velocity 68% local confidence",
)

estimated_velocity_vector = lift(estimated_velocity) do velocity
    if all(isfinite, velocity)
        [Vec2d(velocity[1], velocity[2])]
    else
        [Vec2d(0.0, 0.0)]
    end
end

#=
arrows2d!(
    ax_velocity,
    [Point2d(0.0, 0.0)],
    estimated_velocity_vector;
    color = :orange,
    tipwidth = 12,
    tiplength = 16,
    visible = solution_visible,
)
=#

vlines!(ax_velocity, [0.0]; color = (:gray, 0.5), linewidth = 1)
hlines!(ax_velocity, [0.0]; color = (:gray, 0.5), linewidth = 1)

#=
scatter!(
    ax_velocity,
    initial_velocity_point;
    color = :gray,
    strokecolor = :black,
    strokewidth = 1,
    marker = :diamond,
    markersize = 20,
    label = "Initial velocity guess",
)
=#

# ============================================================
# 表示を常に推定を中心にする
# ============================================================
#=

velocity_range = 10.0

lift(estimated_velocity) do velocity
    limits!(
        ax_velocity,
        velocity[1] - velocity_range,
        velocity[1] + velocity_range,
        velocity[2] - velocity_range,
        velocity[2] + velocity_range,
    )
end


position_range = 10.0

lift(estimated_position) do position
    limits!(
        ax_position,
        position[1] - position_range,
        position[1] + position_range,
        position[2] - position_range,
        position[2] + position_range,
    )
end
=#

# ============================================================
# 数値結果表示
# ============================================================

result_text = lift(
    true_state,
    initial_state,
    noisy_measurements,
    state_solution,
    fdoa_velocity_solution,
    sigma_toa_station,
    sigma_freq_station,
) do truth, initial, measurement, solution, fdoa_solution, sigma_toa, sigma_freq
    if !measurement.valid
        return "Invalid geometry: an object and a station coincide."
    end

    estimated = solution.state

    sigmas = if solution.valid
        sqrt.(max.(diag(solution.covariance), 0.0))
    else
        fill(NaN, 4)
    end

    estimation_error = estimated - truth

    # 2次元68%信頼楕円の長半径・短半径。
    # 共分散楕円(k=1)の半径 sqrt(λ) に、
    # χ²分布（自由度2）の68%点に対応する sqrt(2.279) を掛ける。
    position_ellipse_long_radius_68 = NaN
    position_ellipse_short_radius_68 = NaN
    velocity_ellipse_long_radius_68 = NaN
    velocity_ellipse_short_radius_68 = NaN

    if solution.valid && solution.geometry_reliable
        position_eigenvalues =
            sort(max.(eigen(Symmetric(solution.position_covariance)).values, 0.0))

        velocity_eigenvalues =
            sort(max.(eigen(Symmetric(solution.velocity_covariance)).values, 0.0))

        position_ellipse_short_radius_68 =
            CONFIDENCE_SCALE_68 * sqrt(position_eigenvalues[1])
        position_ellipse_long_radius_68 =
            CONFIDENCE_SCALE_68 * sqrt(position_eigenvalues[end])

        velocity_ellipse_short_radius_68 =
            CONFIDENCE_SCALE_68 * sqrt(velocity_eigenvalues[1])
        velocity_ellipse_long_radius_68 =
            CONFIDENCE_SCALE_68 * sqrt(velocity_eigenvalues[end])
    end

    position_long_radius_text =
        isfinite(position_ellipse_long_radius_68) ?
        @sprintf("%.3f km", position_ellipse_long_radius_68) :
        "ill-conditioned / unobservable"

    position_short_radius_text =
        isfinite(position_ellipse_short_radius_68) ?
        @sprintf("%.3f km", position_ellipse_short_radius_68) :
        "ill-conditioned / unobservable"

    # El = atan(y, x) と定義し、位置共分散を角度空間へ写像する。
    # これは信頼楕円の短半径を距離で割る近似ではなく、
    # 楕円の向きとx-y相関を含めた局所1σである。
    true_el_deg = elevation_angle_deg(truth[1:2])
    estimated_el_deg = elevation_angle_deg(estimated[1:2])

    el_sigma_solution =
        solution.valid ?
        elevation_sigma(estimated[1:2], solution.position_covariance) :
        (
            valid = false,
            sigma_rad = NaN,
            sigma_deg = NaN,
            sigma_arcsec = NaN,
        )

    el_sigma_text =
        el_sigma_solution.valid ?
        @sprintf("%.3f arcsec", el_sigma_solution.sigma_arcsec) :
        "unavailable"

    el_error_arcsec =
        isfinite(true_el_deg) && isfinite(estimated_el_deg) ?
        (estimated_el_deg - true_el_deg) * 3600.0 :
        NaN

    velocity_long_radius_text =
        isfinite(velocity_ellipse_long_radius_68) ?
        @sprintf("%.3f m/s", velocity_ellipse_long_radius_68) :
        "ill-conditioned / unobservable"

    velocity_short_radius_text =
        isfinite(velocity_ellipse_short_radius_68) ?
        @sprintf("%.3f m/s", velocity_ellipse_short_radius_68) :
        "ill-conditioned / unobservable"

    geometry_status =
        solution.geometry_reliable ?
        "reliable local geometry" :
        "ill-conditioned / effectively unobservable"

    fdoa_text = if fdoa_solution.valid
        @sprintf(
            "(%+.3f, %+.3f) m/s, cond(A)=%.3e",
            fdoa_solution.velocity_ms[1],
            fdoa_solution.velocity_ms[2],
            fdoa_solution.condition_number,
        )
    else
        "No unique FDOA-only solution"
    end

    return @sprintf(
        """
        Observation settings
          f₀             = %.3f GHz
          station TDOA σ = %.3f ns
          station FDOA σ = %.3f Hz

        Noisy observations
          TDOA₂₁ = %+.6f μs
          TDOA₃₁ = %+.6f μs
          FDOA₂₁ = %+.6f Hz
          FDOA₃₁ = %+.6f Hz

        Initial guess
          x  = %+.3f km
          y  = %+.3f km
          vx = %+.3f m/s
          vy = %+.3f m/s

        Nonlinear estimate ± local 1σ
          x  = %+.3f ± %.3f km
          y  = %+.3f ± %.3f km
          vx = %+.3f ± %.3f m/s
          vy = %+.3f ± %.3f m/s

        68%% confidence ellipse semi-axes
          position long  = %s
          position short = %s
          velocity long  = %s
          velocity short = %s

        Equivalent angular observation
          El definition   = atan(y, x)
          true El         = %+.6f deg
          estimated El    = %+.6f deg
          σEl (local 1σ)  = %s
          El error        = %+.3f arcsec

        Geometry assessment
          status = %s
          minimum relative singular value = %.3e

        Validation error (estimate - truth)
          dx  = %+.3f km
          dy  = %+.3f km
          dvx = %+.3f m/s
          dvy = %+.3f m/s

        Solver
          valid      = %s
          converged  = %s
          iterations = %d
          cost       = %.6e
          message    = %s
          cond(whitened Jacobian) = %.3e
          position state scale = %.3e km

        FDOA-only velocity at estimated position
          %s
        """,
        F_TX / 1e9,
        sigma_toa * 1e9,
        sigma_freq,
        measurement.tdoa21_s * 1e6,
        measurement.tdoa31_s * 1e6,
        measurement.fdoa21_hz,
        measurement.fdoa31_hz,
        initial[1], initial[2], initial[3], initial[4],
        estimated[1], sigmas[1],
        estimated[2], sigmas[2],
        estimated[3], sigmas[3],
        estimated[4], sigmas[4],
        position_long_radius_text,
        position_short_radius_text,
        velocity_long_radius_text,
        velocity_short_radius_text,
        true_el_deg,
        estimated_el_deg,
        el_sigma_text,
        el_error_arcsec,
        geometry_status,
        minimum(solution.relative_singular_values),
        estimation_error[1],
        estimation_error[2],
        estimation_error[3],
        estimation_error[4],
        string(solution.valid),
        string(solution.converged),
        solution.iterations,
        solution.cost,
        solution.message,
        solution.scaled_information_condition,
        make_state_scale(initial)[1, 1],
        fdoa_text,
    )
end

Label(
    fig[1:4, 5],
    result_text;
    tellwidth = false,
    justification = :left,
    halign = :left,
    valign = :top,
    fontsize = 16,
)


# ============================================================
# 凡例と表示
# ============================================================

Legend(
    fig[3, 1:2],
    ax_position;
    fontsize = 9,
    nbanks = 2,
)

Legend(
    fig[3, 3:4],
    ax_velocity;
    fontsize = 9,
    nbanks = 2,
)

rowgap!(fig.layout, 8)
colgap!(fig.layout, 12)

display(fig)


GLMakie.Screen(...)